# CasADi Models – README Examples

Code snippets from the `casadi-models` README, verified to run.

## Constructing a Nonlinear Model

First, construct state-space model functions, `f` and `h` as CasADi functions in the following form:

```lang-none
    dx(t)/dt = f(t, x(t), u(t))
        y(t) = h(t, x(t), u(t))
```

(for continuous-time models)

In [ ]:
import casadi as cas

# Time variable
t = cas.SX.sym('t')

# States
p = cas.SX.sym('p', 2)
v = cas.SX.sym('v', 2)
x = cas.vertcat(p, v)  # state vector

# Input
u = cas.SX.sym('u', 2)

# Constants
m = cas.DM(0.1)
g = cas.sparsify(cas.DM([0, -9.81]))
c = cas.DM(5.0e-4)
u_nop = cas.DM([0.0, 1.0])

# Righthand side of the ODE
rhs = cas.vertcat(
    v,
    g - c * cas.norm_2(v) * v / m + (u + u_nop) / m
)

f = cas.Function('f', [t, x, u], [rhs], ['t', 'x', 'u'], ['rhs'])
f

In [ ]:
# Construct output function
y = p
h = cas.Function('h', [t, x, u], [y], ['t', 'x', 'u'], ['y'])
h

Pass these functions to the appropriate constructor to build a system model.

In [ ]:
from cas_models.continuous_time.models import StateSpaceModelCT

# System dimensions
n = 4
nu = 2
ny = 2

# Variable names
input_names = ['thrust_x', 'thrust_y']
state_names = ['pos_x', 'pos_y', 'vel_x', 'vel_y']
output_names = ['pos_x', 'pos_y']

model = StateSpaceModelCT(
    f, 
    h, 
    n, 
    nu, 
    ny, 
    name='body',
    input_names=input_names,
    state_names=state_names,
    output_names=output_names
)
model.describe()

## Combine with a Linear Model

In [ ]:
from cas_models.continuous_time.models import SSModelCTLinearFOSISO, SSModelCTLinearFONoGainSISO
from cas_models.transformations import connect_systems_in_series, connect_systems_in_parallel

In [ ]:
# First order linear single-input, single-output (SISO) system
motor_model = SSModelCTLinearFOSISO(K=2.5, T1=0.5, name='motor')
print(motor_model.f)
print(motor_model.h)

In [ ]:
motor_models = connect_systems_in_parallel([motor_model] * 2, model_class=StateSpaceModelCT, name="motors")

# Connect both sub-systems together
sys = connect_systems_in_series([motor_models, model], model_class=StateSpaceModelCT)
sys.describe()

In [ ]:
# Series connections can also be made with the '*' operator
sys = model * motor_models
sys.describe()

## Simulation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from cas_models.discrete_time.models import StateSpaceModelDTFromCT
from cas_models.discrete_time.simulate import make_n_step_simulation_function_from_model

# Convert to discrete-time model with time interval dt
dt = 0.1
sys_dt = StateSpaceModelDTFromCT(sys, dt)
print(sys_dt.F)
print(sys_dt.H)

In [ ]:
# Number of time-steps to simulate
nT = 100

# Create CasADi simulation function
simulate = make_n_step_simulation_function_from_model(sys_dt, nT)
print(simulate)

In [ ]:
# Simulation time
# Note: Simulation outputs include values for nT+1 time instants
t = dt * np.arange(nT + 1)
t_in = t[:-1]

# Simulation inputs
U = np.zeros((nT, sys_dt.nu))
U[t_in >= 1, 0] = 1.0
U[t_in >= 3, 1] = 1.0

# Set parameter value for simulation
T1 = 0.1

# Initial condition
x0 = np.zeros(sys_dt.n)
X, Y = simulate(t, U, x0)

assert X.shape == (nT + 1, sys_dt.n)  # states
assert Y.shape == (nT + 1, sys_dt.ny)  # outputs

In [ ]:
# Make time-series plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(4, 4), sharex=True)
ax1.plot(t, Y[:, 0], label="y1 (pos_x)")
ax1.plot(t, Y[:, 1], label="y2 (pos_y)")
ax1.set_ylabel("y")
ax1.legend()
ax1.set_title("System Outputs")
ax1.grid(False)
ax2.step(t_in, U[:, 0], where="post", label=sys.input_names[0])
ax2.step(t_in, U[:, 1], where="post", label=sys.input_names[1])
ax2.set_xlabel("t")
ax2.set_ylabel("u")
ax2.legend()
ax2.set_title("System Inputs")
ax2.grid(False)
plt.tight_layout()
plt.show()

## Simulating a Feedback Loop

In [ ]:
from cas_models.continuous_time.regulators import SSModelCTPIInt
from cas_models.transformations import connect_feedback_system

# Same PI controller used for both axes so Kc and Ti are shared parameters
pi_ctrl = SSModelCTPIInt(name="pi")
ctrls = connect_systems_in_parallel([pi_ctrl] * 2, model_class=StateSpaceModelCT, name="ctrls")
ctrls.describe()

In [ ]:
# Close the feedback loop around the controllers and plant model
sys_cl = connect_feedback_system(sys * ctrls, model_class=StateSpaceModelCT)

# Add setpoint filters to slow down the response to setpoint changes
spf = SSModelCTLinearFONoGainSISO(name="spf")
setpoint_filters = connect_systems_in_parallel(
    [spf] * 2, 
    model_class=StateSpaceModelCT, 
    name="spfs"
)
sys_cl = sys_cl * setpoint_filters
sys_cl.describe()

In [ ]:
# Convert to discrete-time model and create simulation function
sys_cl_dt = StateSpaceModelDTFromCT(sys_cl, dt)
simulate_cl = make_n_step_simulation_function_from_model(sys_cl_dt, nT)
simulate_cl

In [ ]:
ref_x = np.zeros_like(t)
ref_x[10:] = 1.0
ref_y = np.zeros_like(t)
ref_y[50:] = 1.0

R_full = np.column_stack([ref_x, ref_y])  # shape (nT_cl+1, 2)
R_in = R_full[:-1]                        # reference inputs at nT_cl steps


In [ ]:
# import control as con

# Kp = 400 / 7.5
# #Kp = 0.05

# G = con.tf([Kp], [1, 0])
# print(G)

# t_stop = 10
# t = np.linspace(0, t_stop, 1000)

# Th = abs(1 / Kp)
# Th = 0.1
# print(f"{Th = }")

# Kc = 2 / (Kp * Th)
# Ti = 2 * Th
# print(f"{Kc =}, {Ti = }")
# Gc = con.tf([Kc * Ti, Kc], [Ti, 0])
# print(Gc)

# sys_cl = con.feedback(Gc * G, 1)
# print(sys_cl)

# # Setpoint filter
# F_sp = con.tf([1], [2 * Th, 1])
# print(F_sp)

# sys_cl = sys_cl * F_sp

# sr = con.step_response(sys_cl, T=t)

# plt.figure(figsize=(7, 2.5))
# plt.plot(sr.time, sr.y[0, 0])
# plt.xlabel("Time (s)")
# plt.ylabel("Step Response")
# plt.title("Step Response of Closed-Loop System")
# plt.grid()
# plt.show()

In [ ]:
simulate_cl

In [ ]:
# Estimated process gain (from step response above)
Kp = 400 / 7.5

# Desired closed-loop time constant
Th = abs(1 / Kp)
Th = 0.5

# PI tuning parameters
Kc = 2 / (Kp * Th)
Ti = 2 * Th

# Setpoint filter time constant
T1 = 2 * Th

x0 = np.zeros(sys_cl_dt.n)
assert t.shape == (nT + 1,)
assert R_in.shape == (nT, sys_cl_dt.ny)
X_cl, Y_cl = simulate_cl(t, R_in, x0, T1, Kc, Ti)

assert X_cl.shape == (nT + 1, sys_cl_dt.n)
assert Y_cl.shape == (nT + 1, sys_cl_dt.ny)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(4, 5.5), gridspec_kw={"height_ratios": [3, 1]})

# Plot trajectory in x-y plane
ax = axes[0]
ax.plot(R_full[:, 0], R_full[:, 1], "C0--", label="ref")
ax.plot(Y_cl[:, 0], Y_cl[:, 1], "C1", label="pos")
ax.set_xlim([-1, 2])
ax.set_ylim([-1, 2])
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.legend()
ax.grid(False)

ax = axes[1]
ax.step(t_in, R_in[:, 0], where="post", linestyle="--", label="ref_x")
ax.step(t_in, R_in[:, 1], where="post", linestyle="--", label="ref_y")
ax.set_xlabel("Time (t)")
ax.set_ylabel("u")
ax.legend()
ax.grid(False)

plt.tight_layout()
plt.show()